<a href="https://colab.research.google.com/github/dorhoffman/SWINGPULSE/blob/main/notebooks/08_Testing_and_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd
import yfinance as yf

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"
MODELS_FOLDER = f"{PROJECT_PATH}/models"
UTILS_FOLDER = f"{PROJECT_PATH}/utils"
REPORT_FOLDER = f"{PROJECT_PATH}/report"

MODEL_PATH = f"{MODELS_FOLDER}/random_forest_model.joblib"
METADATA_PATH = f"{MODELS_FOLDER}/model_metadata.json"

os.makedirs(REPORT_FOLDER, exist_ok=True)

print("Model exists:", os.path.exists(MODEL_PATH))
print("Metadata exists:", os.path.exists(METADATA_PATH))
print("Utils exists:", os.path.exists(UTILS_FOLDER))

Mounted at /content/drive
Model exists: True
Metadata exists: True
Utils exists: True


In [2]:
if UTILS_FOLDER not in sys.path:
    sys.path.append(UTILS_FOLDER)

from feature_engineering import add_technical_features

print("Feature engineering module loaded")

Feature engineering module loaded


In [3]:
model = joblib.load(MODEL_PATH)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    metadata = json.load(file)

FEATURE_COLUMNS = metadata["features"]
DECISION_THRESHOLD = metadata["decision_threshold"]
PREDICTION_HORIZON = metadata["prediction_horizon"]
TARGET_RETURN = metadata["target_return"]

print("Model loaded successfully")
print("Features:", len(FEATURE_COLUMNS))
print("Threshold:", DECISION_THRESHOLD)
print("Target:", TARGET_RETURN)
print("Horizon:", PREDICTION_HORIZON)

Model loaded successfully
Features: 19
Threshold: 0.45
Target: 0.1
Horizon: 10


In [4]:
def download_live_stock(
    symbol: str,
    period: str = "2y"
) -> pd.DataFrame:
    symbol = str(symbol).upper().strip()

    data = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False
    )

    if data.empty:
        raise ValueError(
            f"No Yahoo Finance data returned for {symbol}."
        )

    data = data.reset_index()

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data.columns = [
        str(column).strip().replace(" ", "_")
        for column in data.columns
    ]

    if "Adj_Close" in data.columns:
        data = data.drop(columns=["Adj_Close"])

    data["Symbol"] = symbol

    required_columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Yahoo data is missing columns: {missing_columns}"
        )

    return data

In [5]:
def analyze_live_stock(symbol: str) -> dict:
    symbol = str(symbol).upper().strip()

    try:
        live_data = download_live_stock(
            symbol=symbol,
            period="2y"
        )

        live_features = add_technical_features(
            live_data
        )

        valid_rows = live_features.dropna(
            subset=FEATURE_COLUMNS
        )

        if valid_rows.empty:
            return {
                "success": False,
                "symbol": symbol,
                "error_type": "insufficient_data",
                "message": (
                    f"Not enough historical data to calculate "
                    f"all model features for {symbol}."
                )
            }

        latest_row = valid_rows.iloc[-1]

        model_input = pd.DataFrame(
            [latest_row[FEATURE_COLUMNS].to_dict()],
            columns=FEATURE_COLUMNS
        ).replace([np.inf, -np.inf], np.nan)

        if model_input.isna().any().any():
            return {
                "success": False,
                "symbol": symbol,
                "error_type": "invalid_features",
                "message": (
                    f"The latest feature row for {symbol} "
                    "contains invalid values."
                )
            }

        probability = model.predict_proba(
            model_input
        )[0, 1]

        if probability >= DECISION_THRESHOLD + 0.15:
            signal = "Strong Watch"
        elif probability >= DECISION_THRESHOLD:
            signal = "Watch"
        elif probability >= DECISION_THRESHOLD - 0.10:
            signal = "Neutral"
        else:
            signal = "Low Potential"

        return {
            "success": True,
            "symbol": symbol,
            "date": latest_row["Date"],
            "close": float(latest_row["Close"]),
            "probability": float(probability),
            "signal": signal,
            "rsi": float(latest_row["RSI_14"]),
            "macd": float(latest_row["MACD"]),
            "volatility": float(
                latest_row["Volatility_20"]
            )
        }

    except Exception as error:
        return {
            "success": False,
            "symbol": symbol,
            "error_type": "external_or_unknown_error",
            "message": str(error)
        }

In [6]:
valid_result = analyze_live_stock("AAPL")

print(valid_result)

{'success': True, 'symbol': 'AAPL', 'date': Timestamp('2026-07-10 00:00:00'), 'close': 315.32000732421875, 'probability': 0.27028629568588963, 'signal': 'Low Potential', 'rsi': 62.927177747848795, 'macd': 4.614786047137443, 'volatility': 0.0221394778742784}


In [7]:
assert valid_result["success"] is True
assert 0 <= valid_result["probability"] <= 1
assert valid_result["close"] > 0
assert 0 <= valid_result["rsi"] <= 100

print("Valid stock test passed")

Valid stock test passed


In [8]:
invalid_result = analyze_live_stock("NOTREAL")

print(invalid_result)

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTREAL"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTREAL']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


{'success': False, 'symbol': 'NOTREAL', 'error_type': 'external_or_unknown_error', 'message': 'No Yahoo Finance data returned for NOTREAL.'}


In [9]:
assert invalid_result["success"] is False
assert "message" in invalid_result

print("Invalid symbol test passed")

Invalid symbol test passed


In [10]:
result_1 = analyze_live_stock("AAPL")
result_2 = analyze_live_stock("AAPL")

print("Probability 1:", result_1["probability"])
print("Probability 2:", result_2["probability"])

assert result_1["success"] is True
assert result_2["success"] is True

assert np.isclose(
    result_1["probability"],
    result_2["probability"],
    atol=1e-12
)

print("Consistency test passed")

Probability 1: 0.27028629568588963
Probability 2: 0.27028629568588963
Consistency test passed


In [11]:
expected_features = metadata["features"]

assert FEATURE_COLUMNS == expected_features
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS))

print("Feature order test passed")

Feature order test passed


In [12]:
symbols_to_validate = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META"
]

range_test_results = []

for symbol in symbols_to_validate:
    result = analyze_live_stock(symbol)

    range_test_results.append({
        "Symbol": symbol,
        "Success": result["success"],
        "Probability_Valid": (
            0 <= result.get("probability", -1) <= 1
        ),
        "RSI_Valid": (
            0 <= result.get("rsi", -1) <= 100
        ),
        "Close_Valid": (
            result.get("close", 0) > 0
        )
    })

range_test_df = pd.DataFrame(range_test_results)

display(range_test_df)

,Symbol,Success,Probability_Valid,RSI_Valid,Close_Valid
0,AAPL,True,True,True,True
1,MSFT,True,True,True,True
2,NVDA,True,True,True,True
3,AMZN,True,True,True,True
4,META,True,True,True,True


In [13]:
assert range_test_df["Success"].all()
assert range_test_df["Probability_Valid"].all()
assert range_test_df["RSI_Valid"].all()
assert range_test_df["Close_Valid"].all()

print("Indicator range tests passed")

Indicator range tests passed


In [14]:
performance_symbols = [
    "AAPL",
    "MSFT",
    "NVDA"
]

performance_results = []

for symbol in performance_symbols:
    start_time = time.time()

    result = analyze_live_stock(symbol)

    elapsed_time = time.time() - start_time

    performance_results.append({
        "Symbol": symbol,
        "Success": result["success"],
        "Response_Time_Seconds": elapsed_time
    })

performance_df = pd.DataFrame(performance_results)

display(
    performance_df.style.format({
        "Response_Time_Seconds": "{:.2f}"
    })
)

,Symbol,Success,Response_Time_Seconds
0,AAPL,True,0.30
1,MSFT,True,0.28
2,NVDA,True,0.41


In [15]:
test_data = download_live_stock(
    "AAPL",
    period="2y"
)

test_features = add_technical_features(
    test_data
)

latest_complete_row = test_features.dropna(
    subset=FEATURE_COLUMNS
).iloc[-1]

missing_model_features = (
    latest_complete_row[FEATURE_COLUMNS]
    .isna()
    .sum()
)

print(
    "Missing model features in latest row:",
    missing_model_features
)

assert missing_model_features == 0

print("Live feature completeness test passed")

Missing model features in latest row: 0
Live feature completeness test passed


In [16]:
test_cases = [
    {
        "Test_Name": "Valid stock",
        "Symbol": "AAPL",
        "Expected_Success": True
    },
    {
        "Test_Name": "Second valid stock",
        "Symbol": "NVDA",
        "Expected_Success": True
    },
    {
        "Test_Name": "Invalid symbol",
        "Symbol": "NOTREAL",
        "Expected_Success": False
    }
]

validation_results = []

for test_case in test_cases:
    start_time = time.time()

    result = analyze_live_stock(
        test_case["Symbol"]
    )

    elapsed_time = time.time() - start_time

    actual_success = result["success"]

    validation_results.append({
        "Test_Name": test_case["Test_Name"],
        "Symbol": test_case["Symbol"],
        "Expected_Success": test_case[
            "Expected_Success"
        ],
        "Actual_Success": actual_success,
        "Passed": (
            actual_success
            == test_case["Expected_Success"]
        ),
        "Probability": result.get(
            "probability",
            np.nan
        ),
        "Signal": result.get(
            "signal",
            "N/A"
        ),
        "Response_Time_Seconds": elapsed_time,
        "Message": result.get(
            "message",
            ""
        )
    })

validation_results_df = pd.DataFrame(
    validation_results
)

display(
    validation_results_df.style.format({
        "Probability": "{:.4f}",
        "Response_Time_Seconds": "{:.2f}"
    })
)

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTREAL"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTREAL']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


,Test_Name,Symbol,Expected_Success,Actual_Success,Passed,Probability,Signal,Response_Time_Seconds,Message
0,Valid stock,AAPL,True,True,True,0.2703,Low Potential,0.20,
1,Second valid stock,NVDA,True,True,True,0.4598,Watch,0.22,
2,Invalid symbol,NOTREAL,False,False,True,nan,N/A,0.33,No Yahoo Finance data returned for NOTREAL.


In [17]:
assert validation_results_df["Passed"].all()

print("All core validation tests passed successfully")

All core validation tests passed successfully


In [18]:
VALIDATION_RESULTS_PATH = (
    f"{REPORT_FOLDER}/"
    "testing_and_validation_results.csv"
)

PERFORMANCE_RESULTS_PATH = (
    f"{REPORT_FOLDER}/"
    "live_response_time_results.csv"
)

validation_results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)

performance_df.to_csv(
    PERFORMANCE_RESULTS_PATH,
    index=False
)

print("Validation results saved:")
print(VALIDATION_RESULTS_PATH)
print(PERFORMANCE_RESULTS_PATH)

Validation results saved:
/content/drive/MyDrive/SWINGPULSE/report/testing_and_validation_results.csv
/content/drive/MyDrive/SWINGPULSE/report/live_response_time_results.csv


In [19]:
print("=" * 60)
print("SWINGPULSE TESTING SUMMARY")
print("=" * 60)

print(
    f"Core tests passed: "
    f"{validation_results_df['Passed'].sum()}/"
    f"{len(validation_results_df)}"
)

print(
    f"Average response time: "
    f"{performance_df['Response_Time_Seconds'].mean():.2f} "
    "seconds"
)

print(
    "Feature order: Valid"
)

print(
    "Live feature completeness: Valid"
)

print(
    "Invalid-symbol handling: Valid"
)

SWINGPULSE TESTING SUMMARY
Core tests passed: 3/3
Average response time: 0.33 seconds
Feature order: Valid
Live feature completeness: Valid
Invalid-symbol handling: Valid
